In [ ]:
import testing
from testing import batch_size

In [ ]:
testing.batch_size = 20

In [ ]:
testing.batch_size

In [ ]:
import os 
import sys
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
import model
from model import batch_learning_model

local_model = batch_learning_model['cifar10']

In [ ]:
model.batchSize = 20

In [ ]:
model.batchSize

In [ ]:
sums = {20: 50, 30: 60, 70: 60}

In [ ]:
sum([count for placeholder, count in sums.items()])

In [8]:
from data import dataset_train_test, dataset_transform, datasets_labels_count
import torch.nn as nn 
import torch 

train, test, _ = dataset_train_test['mnist']()
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)  # Conv layer 1
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)  # Conv layer 2
        self.pool = nn.MaxPool2d(2)  # Define AvgPool layer
        self.fc1 = nn.Linear(16 * 5 * 5, 120)  # Fully connected layer 1 (corrected)
        self.fc2 = nn.Linear(120, 84)  # Fully connected layer 2
        self.fc3 = nn.Linear(84, num_classes)  # Fully connected output layer

    def forward(self, x):
        x = torch.tanh(self.conv1(x))
        x = self.pool(x) 
        x = torch.tanh(self.conv2(x))
        x = self.pool(x)  
        x = x.view(x.size(0), -1)  
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)  
        return x
    


/home/ckprj2024/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-01 22:12:23.125171: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-01 22:12:23.125199: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-01 22:12:23.125227: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-01 22:12:23.131085: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary i

In [ ]:
m = LeNet()

In [6]:
import torch 
import io



In [5]:
import pickle 
import time 

In [ ]:
st = time.time()
buffer = io.BytesIO(bytearray(10 * 1024 * 1024))
torch.save(m.state_dict(), buffer)
buffer.seek(0)

original_bytes = buffer.getvalue()

received_buffer = io.BytesIO(original_bytes)
# received_state = torch.load(received_buffer)

# m.load_state_dict(received_state)

et = time.time()

print(et - st)


In [4]:
import torch
import io
import time

# Sample model
# m = torch.nn.Linear(10, 5)

st = time.time()

# 🔥 Optimize buffer size
buffer = io.BytesIO(bytearray())  # 10MB buffer

# Serialize
torch.save(m.state_dict(), buffer)
buffer.seek(0)

# 🔥 Avoid extra copying (no .getvalue())
received_buffer = buffer
received_buffer.seek(0)

# Deserialize
# received_state = torch.load(received_buffer)
# m.load_state_dict(received_state)

et = time.time()

print(f"Time taken: {et - st:.4f} seconds")


NameError: name 'm' is not defined

In [326]:
st = time.time()
s_state = pickle.dumps(m.state_dict())
# pickle.loads(s_state)
m.load_state_dict(pickle.loads(s_state))
et = time.time()

print(et - st)

0.21907758712768555


In [ ]:
# import torch
# import io
# import lz4.frame
# import time

# # m = torch.nn.Linear(1000, 500)  # Large model

# st = time.time()

# buffer = io.BytesIO()
# torch.save(m.state_dict(), buffer)

# compressed_bytes = lz4.frame.compress(buffer.getvalue())  # 🔥 Super fast compression
# print(f"Compressed size: {len(compressed_bytes)} bytes")

# # Decompress
# decompressed_bytes = lz4.frame.decompress(compressed_bytes)
# received_buffer = io.BytesIO(decompressed_bytes)
# received_state = torch.load(received_buffer, weights_only=True)

# m.load_state_dict(received_state)

# et = time.time()
# print(f"LZ4 Time: {et - st:.4f} sec")


In [9]:

def conv_block(in_channels, out_channels, pool=False):
    layers = [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            #   CustomBatchNormManualModule(n_neurons=out_channels), 
              nn.BatchNorm2d(out_channels), 
              nn.ReLU(inplace=True)]
    if pool: layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)

class ResNet9(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(ResNet9, self).__init__()
        
        self.conv1 = conv_block(in_channels, 64)
        self.conv2 = conv_block(64, 128, pool=True) 
        self.res1 = nn.Sequential(conv_block(128, 128), conv_block(128, 128)) 
        
        self.conv3 = conv_block(128, 256, pool=True)
        self.conv4 = conv_block(256, 512, pool=True) 
        self.res2 = nn.Sequential(conv_block(512, 512), conv_block(512, 512)) 
        self.conv5 = conv_block(512, 1028, pool=True) 
        self.res3 = nn.Sequential(conv_block(1028, 1028), conv_block(1028, 1028))  
        
        self.classifier = nn.Sequential(nn.MaxPool2d(2), # 1028 x 1 x 1
                                        nn.Flatten(), # 1028 
                                        nn.Linear(1028, num_classes)) # 1028 -> num_classes
        
    def forward(self, xb):
        out = self.conv1(xb)
        out = self.conv2(out)
        out = self.res1(out) + out
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out
        out = self.conv5(out)
        out = self.res3(out) + out
        out = self.classifier(out)
        return out
m = ResNet9(3, 10)


In [ ]:
import torch
# torch.set_num_threads(10)

In [24]:
# import torch
# ok change to this
import io


st = time.time()
# Preallocate buffer (estimate required size)
buffer = io.BytesIO()  # 10MB buffer

# Serialize
# fp16_state_dict = {k: v.half() for k, v in m.state_dict().items()}  
# torch.save(fp16_state_dict, buffer, _use_new_zipfile_serialization=False)

torch.save(m.state_dict(), buffer, _use_new_zipfile_serialization=False)

received_buffer = io.BytesIO(buffer.getvalue())
received_buffer.seek(0)
# buffer.close()
# # Deserialize
received_state = torch.load(received_buffer)

m.load_state_dict(received_state)

et = time.time()

print(et - st)


0.017665863037109375


/tmp/ipykernel_6613/2462197505.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  received_state = torch.load(received_buffer)


In [ ]:
import numpy as np

In [145]:
# ok

import torch
import io

st = time.time()
# Get only weight values
weights_only = list(m.state_dict().values())

# Save weights as a tensor list (no metadata)
buffer = io.BytesIO()
torch.save(weights_only, buffer, _use_new_zipfile_serialization=False)
weights_bytes = buffer.getvalue()  # Get raw bytes

# Simulate sending over network
received_buffer = io.BytesIO(weights_bytes)
loaded_weights = torch.load(received_buffer)  # Load only weight values

state_dict_keys = list(m.state_dict().keys())
new_state_dict = {key: weight for key, weight in zip(state_dict_keys, loaded_weights)}
m.load_state_dict(new_state_dict)

# received_buffer.seek(0)  # Ensure reading from the start

# # Read the raw bytes
# weights_np = np.frombuffer(received_buffer.read(), dtype=np.float32)

et = time.time()

print(et - st)

0.11265301704406738


/tmp/ipykernel_86838/1459021111.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_weights = torch.load(received_buffer)  # Load only weight values


In [1]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        
        weight_decay = 0.0001
        
        # First convolutional block (Conv2D + ReLU + MaxPooling2D + Dropout)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # Conv2D(32, 3x3)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)  # Conv2D(32, 3x3)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)       # MaxPool2D(2x2)
        self.dropout1 = nn.Dropout(p=0.1)

        # Second convolutional block (Conv2D + ReLU + MaxPooling2D + Dropout)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # Conv2D(64, 3x3)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)  # Conv2D(64, 3x3)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)       # MaxPool2D(2x2)
        self.dropout2 = nn.Dropout(p=0.2)

        # Third convolutional block (Conv2D + ReLU + MaxPooling2D + Dropout)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1) # Conv2D(128, 3x3)
        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)# Conv2D(128, 3x3)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)       # MaxPool2D(2x2)
        self.dropout3 = nn.Dropout(p=0.3)

        # Fourth convolutional block (Conv2D + ReLU + MaxPooling2D + Dropout)
        self.conv7 = nn.Conv2d(128, 256, kernel_size=3, padding=1)# Conv2D(256, 3x3)
        self.conv8 = nn.Conv2d(256, 256, kernel_size=3, padding=1)# Conv2D(256, 3x3)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)       # MaxPool2D(2x2)
        self.dropout4 = nn.Dropout(p=0.4)

        # Fully connected layers (FC layer + Softmax)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 2 * 2, 10)  # The size 256*2*2 is derived after all pooling

    def forward(self, x):
        # Forward pass through layers
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool1(x)
        x = self.dropout1(x)
        
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)
        x = self.dropout2(x)

        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))
        x = self.pool3(x)
        x = self.dropout3(x)

        x = F.relu(self.conv7(x))
        x = F.relu(self.conv8(x))
        x = self.pool4(x)
        x = self.dropout4(x)

        # Flatten the output and pass through fully connected layer
        x = self.flatten(x)
        x = self.fc1(x)

        return x


In [2]:
m = CNNModel()

In [5]:
# ok

import torch
import io
import time

st = time.time()
# Get only weight values
weights_only = list(m.state_dict().values())

# Save weights as a tensor list (no metadata)
buffer = io.BytesIO()
torch.save(weights_only, buffer, _use_new_zipfile_serialization=False)
weights_bytes = buffer.getvalue()  # Get raw bytes

# Simulate sending over network
received_buffer = io.BytesIO(weights_bytes)
# loaded_weights = torch.load(received_buffer)  # Load only weight values

# state_dict_keys = list(m.state_dict().keys())
# new_state_dict = {key: weight for key, weight in zip(state_dict_keys, loaded_weights)}
# m.load_state_dict(new_state_dict)

# received_buffer.seek(0)  # Ensure reading from the start

# # Read the raw bytes
# weights_np = np.frombuffer(received_buffer.read(), dtype=np.float32)

et = time.time()

print(et - st)

0.0052220821380615234


In [ ]:
import io
import torch
import numpy as np

st = time.time()
# Convert model state_dict to a flat NumPy array
state_dict = m.state_dict()
flat_weights = np.concatenate([t.cpu().numpy().flatten() for t in state_dict.values()])

# Serialize to bytes
buffer = io.BytesIO()
buffer.write(flat_weights.tobytes())  # Store raw bytes
weights_bytes = buffer.getvalue()

received_buffer = io.BytesIO(weights_bytes)
received_buffer.seek(0)  # Ensure reading from the start

# Convert back to NumPy array
weights_np = np.frombuffer(received_buffer.read(), dtype=np.float32) 

et = time.time()
print(et - st)